In [ ]:
import numpy as np
import joblib

# Load scaler
scaler = joblib.load('../models/scaler.pkl')
print("✓ Scaler loaded")

# Load label encoder
label_encoder = joblib.load('../models/label_encoder.pkl')
print("✓ Label encoder loaded")

# Load PRISM pipeline
prism_pipeline = joblib.load('../models/prism_model.pkl')
print("✓ PRISM loaded")

# The star sample from your raw data
star_raw = {
    'uv': 21.74669,
    'green': 20.03493,
    'red': 19.17553,
    'near_ir': 18.81823,
    'ir': 18.65422,
    'redshift': -0.000008,
    'ug': 1.71176,
    'gr': 0.85940,
    'ri': 0.35730,
    'iz': 0.16401
}

# Create feature array
features = np.array([[
    star_raw['uv'],
    star_raw['green'],
    star_raw['red'],
    star_raw['near_ir'],
    star_raw['ir'],
    star_raw['redshift'],
    star_raw['ug'],
    star_raw['gr'],
    star_raw['ri'],
    star_raw['iz']
]])

# Scale the features
X_scaled = scaler.transform(features)

# Get PRISM prediction
X_enc = prism_pipeline['graph_encoder'].transform(X_scaled)
X_att = prism_pipeline['attention'].transform(X_enc)
prism_pred = prism_pipeline['classifier'].predict(X_att)[0]
prism_proba = prism_pipeline['classifier'].predict_proba(X_att)[0]

print(f"\nPRISM Prediction: {label_encoder.classes_[prism_pred]}")
print(f"Probabilities:")
print(f"  GALAXY: {prism_proba[0]*100:.1f}%")
print(f"  QSO: {prism_proba[1]*100:.1f}%")
print(f"  STAR: {prism_proba[2]*100:.1f}%")

In [ ]:
# Load the cleaned data
df_clean = pd.read_csv('../data/processed/clean_data.csv')

# Get all stars
all_stars = df_clean[df_clean['class'] == 'STAR'].copy()

# Prepare features
feature_cols = ['UV_filter', 'green_filter', 'red_filter', 'near_IR_filter', 
                'IR_filter', 'red_shift', 'u-g', 'g-r', 'r-i', 'i-z']

X_stars = all_stars[feature_cols].values
X_stars_scaled = scaler.transform(X_stars)

# Get PRISM predictions
X_enc_stars = prism_pipeline['graph_encoder'].transform(X_stars_scaled)
X_att_stars = prism_pipeline['attention'].transform(X_enc_stars)
star_preds = prism_pipeline['classifier'].predict(X_att_stars)

# Find stars predicted as STAR
star_label = label_encoder.transform(['STAR'])[0]
correct_stars = all_stars[star_preds == star_label]

print(f"Total stars: {len(all_stars)}")
print(f"Stars PRISM predicts as STAR: {len(correct_stars)}")
print(f"Success rate: {len(correct_stars)/len(all_stars)*100:.1f}%")

if len(correct_stars) > 0:
    sample = correct_stars.iloc[0]
    print(f"uv: {sample['UV_filter']:.5f}")
    print(f"green: {sample['green_filter']:.5f}")
    print(f"red: {sample['red_filter']:.5f}")
    print(f"near_ir: {sample['near_IR_filter']:.5f}")
    print(f"ir: {sample['IR_filter']:.5f}")
    print(f"redshift: {sample['red_shift']:.6f}")
    print(f"ug: {(sample['u-g']):.5f}")
    print(f"gr: {(sample['g-r']):.5f}")
    print(f"ri: {(sample['r-i']):.5f}")
    print(f"iz: {(sample['i-z']):.5f}")